# Task 2 — Trip feature EDA
Run Task 1 and Task 2 public APIs entirely in memory, then inspect segmentation and feature quality. No dataset is written.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
root = next(
    (path for path in (start, *start.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Cannot locate project root containing pyproject.toml")
for import_path in (root, root / "src"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from submission.parser import parse_all
from submission.features import build_trip_features

print(f"Project root: {root}")

Project root: C:\Users\USER\Desktop\mobility-insight_task


In [2]:
bronze = root / "data" / "bronze"
unified_stream = parse_all(
    [str(path) for path in sorted(bronze.glob("tm*.json"))],
    str(bronze / "wdmt.json") if (bronze / "wdmt.json").exists() else "",
    str(bronze / "scgjwd.csv") if (bronze / "scgjwd.csv").exists() else "",
)
trip_features = build_trip_features(unified_stream)
print(f"Events: {len(unified_stream):,} | Trips: {len(trip_features):,}")
display(trip_features.head())

Events: 760 | Trips: 5


,trip_id,source,vehicle_id,trip_start,trip_end,record_count,segmentation_method,trip_end_reason,distance_method,trip_duration_min,...,avg_speed_kmh,max_speed_kmh,speed_variance,harsh_brake_count,harsh_accel_count,idle_ratio,time_of_day_risk,avg_fuel_rate_lh,avg_rpm,data_grade
0,34591f07433a00c5,SCGJWD,001000000111002,2026-03-01 07:00:00+07:00,2026-03-01 07:45:00+07:00,46,engine_and_gap,engine_off,haversine,45.0,...,18.282609,40.0,172.463611,<NA>,<NA>,0.244444,peak,None,None,B
1,636ebab927486fe3,SCGJWD,001000000111003,2026-03-01 07:30:00+07:00,2026-03-01 08:29:00+07:00,60,engine_and_gap,stream_end,haversine,59.0,...,94.516667,110.0,75.416389,<NA>,<NA>,0.000000,peak,None,None,B
2,1c9d8b973980ab4c,WDMT,5กข-1122,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,241,gps_gap_fallback,stream_end,odometer,120.0,...,22.373444,50.0,343.445602,<NA>,<NA>,NaN,peak,None,None,B
3,b6a43e9ff8dd7239,WDMT,6งจ-3344,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,121,gps_gap_fallback,stream_end,odometer,120.0,...,94.396694,110.0,76.520320,<NA>,<NA>,NaN,peak,None,None,B
4,9cddd819efec3559,WDMT,7ดต-5566,2026-03-01 08:00:00+07:00,2026-03-01 09:00:00+07:00,241,gps_gap_fallback,stream_end,haversine,60.0,...,62.033195,90.0,882.380641,<NA>,<NA>,NaN,peak,None,None,B


## Segmentation summary

In [3]:
if not trip_features.empty:
    display(
        trip_features.groupby(
            ["source", "segmentation_method", "trip_end_reason", "data_grade"],
            observed=True,
        )
        .size()
        .rename("trips")
        .to_frame()
    )

trips
source segmentation_method trip_end_reason data_grade       
SCGJWD engine_and_gap      engine_off      B               1
                           stream_end      B               1
WDMT   gps_gap_fallback    stream_end      B               3

## Feature distribution and null coverage

In [4]:
if not trip_features.empty:
    numeric = [
        "trip_duration_min",
        "distance_km",
        "avg_speed_kmh",
        "max_speed_kmh",
        "speed_variance",
        "idle_ratio",
        "avg_fuel_rate_lh",
        "avg_rpm",
    ]
    display(trip_features[numeric].describe().transpose())
    display((trip_features.isna().mean() * 100).round(2).rename("null_pct").to_frame())

,count,mean,std,min,25%,50%,75%,max
trip_duration_min,5.0,80.800000,36.272579,45.000000,59.000000,60.000000,120.000000,120.000000
distance_km,5.0,3.778108,3.165215,0.749000,2.729236,3.102738,3.151000,9.158566
avg_speed_kmh,5.0,58.320522,37.150579,18.282609,22.373444,62.033195,94.396694,94.516667
max_speed_kmh,5.0,80.000000,33.166248,40.000000,50.000000,90.000000,110.000000,110.000000
speed_variance,5.0,310.045312,338.081605,75.416389,76.520320,172.463611,343.445602,882.380641
idle_ratio,2.0,0.122222,0.172848,0.000000,0.061111,0.122222,0.183333,0.244444


,null_pct
trip_id,0.0
source,0.0
vehicle_id,0.0
trip_start,0.0
trip_end,0.0
record_count,0.0
segmentation_method,0.0
trip_end_reason,0.0
distance_method,0.0
trip_duration_min,0.0
